In [5]:
import json

with open('../data/task3.json', 'r') as f:
    data = json.load(f)

test_texts = data['drafts']
test_labels = data['labels']

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. Load environment variables from the .env file
load_dotenv()

# 2. Safely retrieve the API Key
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")

# Add a safety check to ensure the API key is loaded properly
if not SILICONFLOW_API_KEY:
    raise ValueError("API Key not found. Please ensure your .env file is created correctly and contains the SILICONFLOW_API_KEY variable!")

# 3. Configure the LLM using the SiliconFlow API
llm = ChatOpenAI(
    model="Pro/moonshotai/Kimi-K2.5", 
    api_key=SILICONFLOW_API_KEY,
    base_url="https://api.siliconflow.cn/v1", 
    temperature=0.0,
    max_tokens=5 # Keep this to restrict the output length as in the original code
)

In [7]:
import random
from tqdm import tqdm
from langchain_core.messages import HumanMessage, SystemMessage

def classify_texts(texts):
    results = []
    
    # Iterate through the texts with a progress bar
    for text in tqdm(texts):
        user_prompt = f"""
        The provided document is a United Nations Security Council's draft resolution. Predict whether the draft resolution will be adopted or not. Answer with 'yes' (1) or 'no' (0) without any explanation.

        Text: "{text}"
        Answer:
        """
        
        try:
            # Construct LangChain messages and invoke the model
            messages = [
                SystemMessage(content="You are a helpful assistant."),
                HumanMessage(content=user_prompt)
            ]
            response = llm.invoke(messages)
            
            # Clean and lowercase the output for exact matching
            result = response.content.strip().lower()
            
            # Strictly use the original matching logic:
            if result.startswith("yes") or result == "1":
                results.append(1)
            elif result.startswith("no") or result == "0":
                results.append(0)
            else:
                # Fallback to a random choice if the model outputs something unexpected
                results.append(random.choice([0, 1]))  
                
        except Exception as e:
            # Catch API errors (e.g., network issues) and use a random choice to prevent crashing
            print(f"Error generating classification: {e}")
            results.append(random.choice([0, 1]))
            
    return results

# Execute the function and store predictions
pred = classify_texts(test_texts)

100%|██████████| 30/30 [12:56<00:00, 25.90s/it]


In [8]:
# calculate metrics
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, precision_recall_curve, auc

def calculate_metrics(pred, labels):
    # swap 0 and 1
    pred = [1 - x for x in pred]
    labels = [1 - x for x in labels]
    acc = accuracy_score(labels, pred)
    try:
        roc_auc = roc_auc_score(labels, pred)
    except ValueError:
        roc_auc = 0
    balanced_acc = balanced_accuracy_score(labels, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, pred, average='binary')
    # pr_auc = average_precision_score(labels, pred)
    precision, recall, _ = precision_recall_curve(labels, pred)
    pr_auc = auc(recall, precision)
    mcc = matthews_corrcoef(labels, pred)
    g_mean = geometric_mean_score(labels, pred)
    tn, fp, fn, tp = confusion_matrix(labels, pred).ravel()
    specificity = tn / (tn + fp)

    print(f'Accuracy: {acc}')
    print(f'AUC: {roc_auc}')
    print(f'Balanced Accuracy: {balanced_acc}')
    print(f'Precision: {prec}')
    print(f'Recall: {rec}')
    print(f'F1: {f1}')
    print(f'PR AUC: {pr_auc}')
    print(f'MCC: {mcc}')
    print(f'G-Mean: {g_mean}')
    print(f'Specificity: {specificity}')

    print('Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean Specificity')
    print(f'{acc:.4f} {roc_auc:.4f} {balanced_acc:.4f} {prec:.4f} {rec:.4f} {f1:.4f} {pr_auc:.4f} {mcc:.4f} {g_mean:.4f} {specificity:.4f}')



In [9]:
calculate_metrics(pred, test_labels)

Accuracy: 1.0
AUC: 1.0
Balanced Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
PR AUC: 1.0
MCC: 1.0
G-Mean: 1.0
Specificity: 1.0
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean Specificity
1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000
